## Micrograd

In [883]:
import math

class Value:

    def __init__(self, data, _children = (), op = (), label = " "):
        self.data = data
        self.grad = 0
        self._backward = lambda: None 
        self._prev = set(_children)
        self._op = op
        self._label = label
    
    def __repr__(self):
        return f"Value(label={self._label}, data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward 
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward 
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * (other**-1)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float powers are supported for now"
        out = Value(self.data ** other, (self,), f"**{other}")

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1)/(math.exp(2*n) + 1)
        out = Value(t, (self, ), label="tanh")

        def _backward():
            self.grad +=  (1 - t**2) * out.grad
        
        out._backward = _backward 
        return out

    def backprop(self):
        self._backward()
        # print(f"label: {self._label}, grad: {self.grad}")
        for child in self._prev:
            child.backprop()

In [884]:
a = Value(3.0, label="a")
b = a + a
b.grad = 1
b.backprop()

In [885]:
a + 1
a * 2
2 * a
2 + a

Value(label= , data=5.0)

In [886]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.8813735870195432, label="b")

x1w1 = x1*w1
x1w1._label = "x1*w1"

x2w2 = x2*w2
x2w2._label = "x2*w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2._label = "w1*w1 + x2*w2"

n = x1w1x2w2 + b
n._label = "n"

o = n.tanh()
o._label = "o"

In [887]:
o.grad = 1.0

In [888]:
o.backprop()

In [889]:
import random

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for i in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        assert len(x) == len(self.w), "number of inputs is not equal to the number of weights"
        act = sum([xi * self.w[i] for i, xi in enumerate(x)]) + self.b
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [890]:
x = [2.0, 3.0]
n = Neuron(len(x))
n(x)

Value(label=tanh, data=0.9997368377020021)

In [891]:
class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [892]:
x = [2.0, 3.0]
la = Layer(len(x), 3)
la(x)

[Value(label=tanh, data=-0.6506131623019044),
 Value(label=tanh, data=-0.5878899214609044),
 Value(label=tanh, data=0.9949069729859242)]

In [893]:
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        out = x
        for layer in self.layers:
            out = layer(out)
        return out

    def parameters(self):
        return [p for la in self.layers for p in la.parameters()]

In [894]:
x = [2.0, 3.0, -1.0]
mlp = MLP(len(x), [4, 4, 1])
mlp(x)

Value(label=tanh, data=-0.6174491654029063)

In [895]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

In [896]:
ypreds = [mlp(x) for x in xs]
ypreds

[Value(label=tanh, data=-0.6174491654029063),
 Value(label=tanh, data=-0.5070747536069381),
 Value(label=tanh, data=-0.8452590442416357),
 Value(label=tanh, data=-0.7203497575753474)]

In [897]:
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss._label = "loss"
loss

Value(label=loss, data=5.842665152972787)

In [898]:
loss.grad = 1.0
loss.backprop()

In [899]:
len(mlp.parameters())

41

In [900]:
STEP = 0.01

for p in mlp.parameters():
    p_old = p.data
    p.data += p.grad * (-STEP)
    print(f"parameter updated from {p_old} to {p.data}")

parameter updated from 0.6053179864422338 to -0.06212467416514911
parameter updated from -0.14337040170963067 to -0.5353785425549398
parameter updated from 0.4166703511474674 to 0.6061599612570612
parameter updated from -0.6010113849473113 to -0.6561013243471968
parameter updated from -0.5257811841623938 to -0.6108246421144239
parameter updated from -0.13941240916152742 to -0.19738048539107475
parameter updated from -0.09605162140307155 to -0.058616447419786506
parameter updated from -0.9388817937404033 to -0.9489869893560995
parameter updated from 0.3605652139419151 to 0.7349421453455798
parameter updated from 0.8693828303230615 to 0.785612486162753
parameter updated from -0.07250862001201508 to -0.04662780048808467
parameter updated from 0.7865097400401779 to 0.8006026391635657
parameter updated from 0.13161638635809436 to 0.01655589866675425
parameter updated from -0.7793075555772055 to -0.866641683382682
parameter updated from 0.9761977731485401 to 1.0275833902938736
parameter upda

In [901]:
TRAINING_ITER = 100

def train(mlp, xs, ys):
    for i in range(TRAINING_ITER):
        # forward
        ypreds = [mlp(x) for x in xs]
        loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
        print(f"loss {loss}")
        loss._label = "loss"
        loss.grad = 1
        for p in mlp.parameters():
            p.grad = 0.0
        # backward
        loss.backprop()
        # update
        for p in mlp.parameters():
            p.data += p.grad * (-STEP)

        
    

In [902]:
train(mlp, xs, ys)

ypreds = [mlp(x) for x in xs]
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss

loss Value(label= , data=5.648909298841408)
loss Value(label= , data=4.997051665918461)
loss Value(label= , data=4.149694712564308)
loss Value(label= , data=3.2718268707485088)
loss Value(label= , data=2.4862091096381524)
loss Value(label= , data=1.9011957538795012)
loss Value(label= , data=1.53085218208129)
loss Value(label= , data=1.3003599448337737)
loss Value(label= , data=1.0885213081517666)
loss Value(label= , data=0.9597398856653014)
loss Value(label= , data=0.8489614008800264)
loss Value(label= , data=0.7602836000891812)
loss Value(label= , data=0.6710920482551805)
loss Value(label= , data=0.6022876393038217)
loss Value(label= , data=0.5375731000397261)
loss Value(label= , data=0.4733276775955739)
loss Value(label= , data=0.443192748868246)
loss Value(label= , data=0.39050983482010615)
loss Value(label= , data=0.35835363256914055)
loss Value(label= , data=0.37783677466620524)
loss Value(label= , data=0.41696576136864616)
loss Value(label= , data=0.37639048899701594)
loss Value(

Value(label= , data=0.020580229358471716)